This simple notebook helps you find the experiment that you want, as the timestamps are not so useful.

In [ ]:
import os
import json
from multiprocessing import Pool
import pandas as pd
from ipywidgets import interact, widgets, interactive
from IPython.display import display

In [ ]:
def find_config_files(results_dir):
    """
    Scans the results directory and returns a list of paths to all config.json files.
    """
    config_paths = []
    for root, dirs, files in os.walk(results_dir):
        if 'config.json' in files:
            config_paths.append(os.path.join(root, 'config.json'))
    return config_paths

def load_config(file_path):
    """
    Loads a single config.json file and extracts the experiment ID.
    """
    try:
        with open(file_path, 'r') as f:
            config_data = json.load(f)
            # The experiment ID is the name of the parent directory of the config file
            experiment_id = os.path.basename(os.path.dirname(file_path))
            config_data['experiment_id'] = experiment_id
            return config_data
    except json.JSONDecodeError:
        print(f"Warning: Could not decode JSON from {file_path}")
        return None
    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}")
        return None

def load_all_configs(results_dir):
    """
    Finds all config.json files and loads them in parallel.
    """
    config_files = find_config_files(results_dir)
    if not config_files:
        print("No 'config.json' files found in the specified directory.")
        return []

    # Use multiprocessing to load files in parallel
    with Pool() as pool:
        all_configs = pool.map(load_config, config_files)

    # Filter out any configs that failed to load
    return [config for config in all_configs if config is not None]

# --- Load the data ---
print("Scanning for experiment configurations...")
all_experiment_configs = load_all_configs('.')

if all_experiment_configs:
    print(f"Successfully loaded {len(all_experiment_configs)} experiment configurations.")
    # Create a pandas DataFrame for easier filtering
    df_configs = pd.DataFrame(all_experiment_configs)
else:
    print("Could not load any experiment configurations.")
    df_configs = pd.DataFrame()

In [ ]:
import collections

def create_filter_widgets(df):
    """
    Creates a dictionary of interactive widgets, including a special text filter
    for the experiment_id.
    """
    widgets_dict = collections.OrderedDict() # Use an ordered dict to control widget display order
    style = {'description_width': 'initial'}

    widgets_dict['experiment_id_contains'] = widgets.Text(
        value='',
        placeholder='e.g., 2025_10_16',
        description='Experiment ID Contains:',
        disabled=False,
        style=style
    )

    # Now, create the dropdowns for other relevant columns
    potential_filter_keys = [
        col for col in df.columns
        if df[col].dtype == 'object' and col not in ['experiment_id', 'output_file_name', 'timestamp']
    ]

    for key in potential_filter_keys:
        # Convert all unique values to strings to prevent sorting errors with mixed types
        unique_values = df[key].unique()
        options = ['All'] + sorted([str(v) for v in unique_values])
        widgets_dict[key] = widgets.Dropdown(options=options, value='All', description=key, style=style)
    return widgets_dict

if not df_configs.empty:
    filter_widgets = create_filter_widgets(df_configs)

    def filter_experiments(**filters):
        """
        Filters the DataFrame based on the selected widget values and displays the results.
        """
        filtered_df = df_configs.copy()
        display_columns = ['experiment_id'] # Start with the essential column

        id_contains_value = filters.get('experiment_id_contains', '').strip()
        if id_contains_value:
            # Apply the filter for experiment_id using string contains
            # na=False ensures that any missing experiment_ids don't cause an error
            filtered_df = filtered_df[filtered_df['experiment_id'].str.contains(id_contains_value, na=False)]

        # Handle the dropdown filters
        for key, value in filters.items():
            if key == 'experiment_id_contains': # We already handled this
                continue

            if value != 'All':
                # Ensure we compare strings to strings for consistency
                filtered_df = filtered_df[filtered_df[key].astype(str) == value]

            # Add the filtered column to our display list
            if key not in display_columns:
                display_columns.append(key)

        # Display the matching experiment IDs and the columns being filtered on
        print(f"Found {len(filtered_df)} matching experiments.")
        display(filtered_df[display_columns])

    # The main interactive component
    # interact(filter_experiments, **filter_widgets)
    ui = interactive(filter_experiments, **filter_widgets)
    display(ui)
else:
    print("DataFrame is empty, cannot create interactive widgets.")